In [1]:
import pandas as pd
import autogen
import re
from tqdm import tqdm

# 1. Setup LLM Lokal (Ollama)
local_llm_setup = {
    "config_list": [
        {
            "model": "llama3",
            "base_url": "http://localhost:11434/v1",
            "api_key": "dummy-key-ollama" # Wajib ada format api_key
        }
    ],
    "temperature": 0.1
}

# 2. Agen Pertama: Penilai Utama
agent_assessor = autogen.AssistantAgent(
    name="Primary_Assessor",
    system_message="""Anda bertugas sebagai Guru Evaluator. Baca esai yang diberikan, berikan ringkasan evaluasi (maksimal 3 kalimat) terkait kelancaran bahasa dan substansi. 
    Di akhir kalimat, sebutkan estimasi skor dari skala 1 sampai 6.""",
    llm_config=local_llm_setup,
)

# 3. Agen Kedua: Pengambil Keputusan
agent_final_judge = autogen.AssistantAgent(
    name="Final_Judge",
    system_message="""Peran Anda adalah Kepala Penilai. Analisis argumen dari Primary_Assessor. 
    Anda bebas setuju atau tidak. KETENTUAN WAJIB: Balasan Anda HANYA boleh berisi SATU KARAKTER ANGKA (1, 2, 3, 4, 5, atau 6) yang merepresentasikan skor mutlak. Jangan tambahkan kata apapun.""",
    llm_config=local_llm_setup,
)

# 4. Agen Ketiga: Koordinator Diskusi
agent_coordinator = autogen.UserProxyAgent(
    name="Coordinator",
    human_input_mode="NEVER", 
    max_consecutive_auto_reply=1,
    code_execution_config=False,
    is_termination_msg=lambda msg: "TERMINATE" in msg.get("content", "")
)

print("Semua agen AI siap beroperasi!")

Semua agen AI siap beroperasi!


In [2]:
def execute_agent_debate(essay_content):
    # Reset memori obrolan agar agen tidak tercampur dengan esai sebelumnya
    agent_coordinator.clear_history()
    agent_assessor.clear_history()
    agent_final_judge.clear_history()
    
    # Inisiasi wadah diskusi
    discussion_forum = autogen.GroupChat(
        agents=[agent_coordinator, agent_assessor, agent_final_judge], 
        messages=[], 
        max_round=3 # Alur: Coordinator -> Assessor -> Final Judge
    )
    
    chat_manager = autogen.GroupChatManager(groupchat=discussion_forum, llm_config=local_llm_setup)
    
    # Perintah awal
    prompt_task = f"Mohon berikan evaluasi dan skor untuk esai berikut: \n\n{essay_content}"
    
    # Eksekusi obrolan
    agent_coordinator.initiate_chat(chat_manager, message=prompt_task)
    
    # Ambil output dari agen terakhir (Final_Judge)
    final_response = agent_coordinator.chat_messages[chat_manager][-1]['content']
    
    # Ekstraksi menggunakan Regex untuk menangkap angka saja
    extracted_score = re.search(r'[1-6]', final_response)
    if extracted_score:
        return int(extracted_score.group())
    return 3 # Skor default jika terjadi error format

In [3]:
# Pastikan path direktori sudah benar
train_data = pd.read_csv('../dataset/train.csv').head(5) 

agent_predictions = []

print("Memulai evaluasi Multi-Agent (5 Sampel)...")

for idx, data_row in tqdm(train_data.iterrows(), total=train_data.shape[0]):
    text_essay = data_row['full_text']
    
    # Panggil fungsi diskusi
    assigned_score = execute_agent_debate(text_essay)
    agent_predictions.append(assigned_score)

train_data['multi_agent_pred'] = agent_predictions

print("\n--- Hasil Simulasi ---")
display(train_data[['essay_id', 'score', 'multi_agent_pred']])

Memulai evaluasi Multi-Agent (5 Sampel)...


  0%|          | 0/5 [00:00<?, ?it/s]

Coordinator (to chat_manager):

Mohon berikan evaluasi dan skor untuk esai berikut: 

Many people have car where they live. The thing they don't know is that when you use a car alot of thing can happen like you can get in accidet or the smoke that the car has is bad to breath on if someone is walk but in VAUBAN,Germany they dont have that proble because 70 percent of vauban's families do not own cars,and 57 percent sold a car to move there. Street parkig ,driveways and home garages are forbidden on the outskirts of freiburd that near the French and Swiss borders. You probaly won't see a car in Vauban's streets because they are completely "car free" but If some that lives in VAUBAN that owns a car ownership is allowed,but there are only two places that you can park a large garages at the edge of the development,where a car owner buys a space but it not cheap to buy one they sell the space for you car for $40,000 along with a home. The vauban people completed this in 2006 ,they said that

 20%|██        | 1/5 [00:41<02:44, 41.02s/it]

Coordinator (to chat_manager):

Mohon berikan evaluasi dan skor untuk esai berikut: 

I am a scientist at NASA that is discussing the "face" on mars. I will be explaining how the "face" is a land form. By sharing my information about this isue i will tell you just that.

First off, how could it be a martions drawing. There is no plant life on mars as of rite now that we know of, which means so far as we know it is not possible for any type of life. That explains how it could not be made by martians. Also why and how would a martion build a face so big. It just does not make any since that a martian did this.

Next, why it is a landform. There are many landforms that are weird here in America, and there is also landforms all around the whole Earth. Many of them look like something we can relate to like a snake a turtle a human... So if there are landforms on earth dont you think landforms are on mars to? Of course! why not? It's just unique that the landform on Mars looks like a human f

...

 80%|████████  | 4/5 [02:15<00:33, 33.48s/it]

Coordinator (to chat_manager):

Mohon berikan evaluasi dan skor untuk esai berikut: 

Dear, State Senator

This is a letter to argue in favor of keeping the Electoral College."There are many reasons to keep the Electoral College" one reason is because it is widely regarded as an anachronism, a dispute over the outcome of an Electoral College vote is possible, but it is less likely than a dispute over the popular vote, and the Electoral College restores some of the weight in the political balance that large states (by population) lose by virue of the mal apportionment of the Senate decreed in the Constitution.

I am in favor of keeping the Electoral College because,it is widely regarded as an anachronism. A non-democratic method of selecting a president that ought to be [overruled] by declaring the canaditdate who receives the most populare votes the winner. The advocates of this position are correct in arguing that the Electoral College method is not democratic in a method sense.It is 

100%|██████████| 5/5 [02:52<00:00, 34.53s/it]


--- Hasil Simulasi ---


,essay_id,score,multi_agent_pred
0,000d118,3,4
1,000fe60,3,5
2,001ab80,4,3
3,001bdc0,4,4
4,002ba53,3,3


In [4]:
# 1. Load dataset uji
test_data = pd.read_csv('../dataset/test.csv')

final_test_predictions = []

print("Memproses data test Kaggle menggunakan Multi-Agent...")

# 2. Iterasi penilaian
for idx, data_row in tqdm(test_data.iterrows(), total=test_data.shape[0]):
    target_text = data_row['full_text']
    
    # Hitung skor
    final_score = execute_agent_debate(target_text)
    final_test_predictions.append(final_score)

# 3. Bentuk DataFrame sesuai requirement Kaggle
submission_df = pd.DataFrame({
    'essay_id': test_data['essay_id'],
    'score': final_test_predictions
})

# 4. Ekspor hasil
submission_df.to_csv('../outputs/submission_autogen.csv', index=False)

print("\nBerhasil! File 'submission_autogen.csv' siap diunggah ke Kaggle.")
display(submission_df.head())

Memproses data test Kaggle menggunakan Multi-Agent...


  0%|          | 0/3 [00:00<?, ?it/s]

Coordinator (to chat_manager):

Mohon berikan evaluasi dan skor untuk esai berikut: 

Many people have car where they live. The thing they don't know is that when you use a car alot of thing can happen like you can get in accidet or the smoke that the car has is bad to breath on if someone is walk but in VAUBAN,Germany they dont have that proble because 70 percent of vauban's families do not own cars,and 57 percent sold a car to move there. Street parkig ,driveways and home garages are forbidden on the outskirts of freiburd that near the French and Swiss borders. You probaly won't see a car in Vauban's streets because they are completely "car free" but If some that lives in VAUBAN that owns a car ownership is allowed,but there are only two places that you can park a large garages at the edge of the development,where a car owner buys a space but it not cheap to buy one they sell the space for you car for $40,000 along with a home. The vauban people completed this in 2006 ,they said that

 33%|███▎      | 1/3 [00:40<01:20, 40.48s/it]

Coordinator (to chat_manager):

Mohon berikan evaluasi dan skor untuk esai berikut: 

I am a scientist at NASA that is discussing the "face" on mars. I will be explaining how the "face" is a land form. By sharing my information about this isue i will tell you just that.

First off, how could it be a martions drawing. There is no plant life on mars as of rite now that we know of, which means so far as we know it is not possible for any type of life. That explains how it could not be made by martians. Also why and how would a martion build a face so big. It just does not make any since that a martian did this.

Next, why it is a landform. There are many landforms that are weird here in America, and there is also landforms all around the whole Earth. Many of them look like something we can relate to like a snake a turtle a human... So if there are landforms on earth dont you think landforms are on mars to? Of course! why not? It's just unique that the landform on Mars looks like a human f

 67%|██████▋   | 2/3 [01:12<00:35, 35.39s/it]

Coordinator (to chat_manager):

Mohon berikan evaluasi dan skor untuk esai berikut: 

People always wish they had the same technology that they have seen in movies, or the best new piece of technology that is all over social media. However, nobody seems to think of the risks that these kinds of new technologies may have. Cars have been around for many decades, and now manufacturers are starting to get on the bandwagon and come up with the new and improved technology that they hope will appeal to everyone. As of right now, it seems as though the negative characteristics of these cars consume the positive idea that these manufacturers have tried to convey.

Currently, this new technology in cars has a very long way to go before being completely "driverless". Drivers still need to be on alert when they are driving, as well as control the car near any accidents or complicated traffic situations. This seems to totally defeat the purpose of the "driverless" car. Eventually the technology may

100%|██████████| 3/3 [01:38<00:00, 32.94s/it]


Berhasil! File 'submission_autogen.csv' siap diunggah ke Kaggle.


,essay_id,score
0,000d118,4
1,000fe60,5
2,001ab80,3
